In [111]:
##### Compiles rasters of all predictors into one file

import os
import pandas as pd
import rasterio
from pathlib import Path
import numpy as np
from scipy.spatial import cKDTree
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Transformer

In [112]:
### Set-up

# Get the current working directory
cd = Path.cwd().parent 

# folder of predictors 
predictors = Path(f'{cd}/Data/Clean/Predictors/Rasters')
tif_files  = sorted(predictors.glob("*.tif"))

# USD production 
production_path_USD = f'{cd}/Data/Clean/Production/total_production_USD_2020.tif'
production_path_tonnes = f'{cd}/Data/Clean/Production/total_production_tonnes_2020.tif'

# country averages
country_avg = pd.read_csv(f'{cd}/Data/Clean/Predictors/country_averages.csv')

# correspondance table
country_names = pd.read_csv(f'{cd}/Data/Correspondence_tables/country_names.csv', encoding="cp1252")

# country geographies 
countries = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer="ADM_0")

# save path
save_path_predictors_abs = f'{cd}/Data/Clean/Training_data/raster_predictor_matrix_absolute.parquet'
save_path_predictors_rtv = f'{cd}/Data/Clean/Training_data/raster_predictor_matrix_relative.parquet'

### Create absolute training dataset

In [113]:
#### Combine all rasters

# read rasters
arrays = {}
coords = None
reference_shape = None

for path in tif_files:
    with rasterio.open(path) as src:
        arr    = src.read(1).astype(np.float32)
        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan

        if coords is None:
            reference_shape = arr.shape
            height, width = arr.shape
            transform = src.transform
            raster_crs = src.crs
            rows, cols = np.meshgrid(np.arange(height), np.arange(width), indexing="ij")
            xs, ys = rasterio.transform.xy(transform, rows.ravel(), cols.ravel())
            xs = np.array(xs)
            ys = np.array(ys)

            # convert pixel centroids to lat/lon if not already geographic
            if raster_crs.to_epsg() == 4326:
                lon, lat = xs, ys
            else:
                transformer = Transformer.from_crs(raster_crs, "EPSG:4326", always_xy=True)
                lon, lat = transformer.transform(xs, ys)

            coords = pd.DataFrame({"x": xs, "y": ys, "lon": lon, "lat": lat})

        else:
            if arr.shape != reference_shape:
                raise ValueError(
                    f"Shape mismatch: {path.name} has shape {arr.shape}, "
                    f"expected {reference_shape}"
                )

        arrays[path.stem] = arr.ravel()

# add production rasters
for label, prod_path in [("total_production_USD_2020", production_path_USD),
                          ("total_production_tonnes_2020", production_path_tonnes)]:
    with rasterio.open(prod_path) as src:
        arr    = src.read(1).astype(np.float32)
        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan
        if arr.shape != reference_shape:
            raise ValueError(
                f"Shape mismatch: {label} has shape {arr.shape}, "
                f"expected {reference_shape}"
            )
        arrays[label] = arr.ravel()

# compile into dataframe
all_rasters = pd.concat([coords, pd.DataFrame(arrays)], axis=1)

# drop if terrain is missing (selects only land pixels)
all_rasters = all_rasters.dropna(subset=["terrain_slope"])
all_rasters = all_rasters.dropna(subset=["total_production_USD_2020"])
all_rasters = all_rasters[all_rasters['total_production_USD_2020'] != 0]

In [114]:
### Fill in missing data 

# define function which fill's NaN's with value from nearest pixel that has data (by Euclidean distance in x/y space)
def fill_missing_nearest(df, coord_cols=("x", "y")):
    
    data_cols = [c for c in df.columns if c not in coord_cols]
    coords_arr = df[list(coord_cols)].values

    for col in data_cols:
        n_missing = df[col].isna().sum()
        if n_missing == 0:
            continue

        print(f"  Filling {n_missing:,} missing values in '{col}'...")

        has_data = ~df[col].isna()

        # build tree from pixels that have data for this column
        tree = cKDTree(coords_arr[has_data])

        # query tree for each missing pixel
        missing_mask = ~has_data
        _, idx = tree.query(coords_arr[missing_mask], workers=-1)

        # map back to donor values
        donor_values = df.loc[has_data, col].values[idx]
        df.loc[missing_mask, col] = donor_values

    return df

all_rasters = fill_missing_nearest(all_rasters, coord_cols=("x", "y", "lon", "lat"))

  Filling 39 missing values in 'GDP_per_capita'...
  Filling 2,420 missing values in 'USD_production_per_HA'...
  Filling 6 missing values in 'child_dependency_ratio'...
  Filling 20,388 missing values in 'country_capital_intensity_USD'...
  Filling 20,388 missing values in 'country_capital_intensity_tonnes'...
  Filling 31 missing values in 'country_labor_intensity_USD'...
  Filling 31 missing values in 'country_labor_intensity_tonnes'...
  Filling 6 missing values in 'crop_intensity'...
  Filling 258 missing values in 'economic_objective_prob'...
  Filling 6 missing values in 'female_share'...
  Filling 4,323 missing values in 'field_size_share_large'...
  Filling 4,323 missing values in 'field_size_share_medium'...
  Filling 4,323 missing values in 'field_size_share_small'...
  Filling 4,323 missing values in 'field_size_share_vlarge'...
  Filling 4,323 missing values in 'field_size_share_vsmall'...
  Filling 2,420 missing values in 'log_USD_production_per_HA'...
  Filling 20,388 mi

In [115]:
#### Drop compositional variables

# one of each of field size %'s and production mix %'s 
# share_vlarge_field, other_share_production_USD  

# also split predictors and target

col_to_drop = ['field_size_share_vlarge', 'other_share_production_USD']
predictors = all_rasters.drop(columns = col_to_drop)

In [116]:
##### Rename column names to match vectors

rename_dict = {
    'soil_organic_carbon': 'SOC',
    'pop_density': 'pop_density_people_per_km2',
    'travel_time_city': 'average_travel_time_city',
    'travel_time_port': 'average_travel_time_port',
    'economic_objective_prob': 'probability_economic_land_use_objective',
    'survival_objective_prob': 'probability_survival_land_use_objective',
    'season_length': 'growing_season_length_days',
    'GDP_per_capita': 'GDP_pc',
    'terrain_slope': 'slope',
    'livestock_density_buffalo_per_km2': 'buffalo_density_per_km2',
    'livestock_density_chicken_per_km2': 'chicken_density_per_km2',
    'livestock_density_cattle_per_km2': 'cattle_density_per_km2',
    'livestock_density_goats_per_km2': 'goats_density_per_km2',
    'livestock_density_pigs_per_km2': 'pigs_density_per_km2',
    'livestock_density_sheep_per_km2': 'sheep_density_per_km2',
    'field_size_share_large': 'share_large_field',
    'field_size_share_medium': 'share_medium_field',
    'field_size_share_small': 'share_small_field',
    'field_size_share_vsmall': 'share_vsmall_field',
    'total_production_USD_2020': 'total_production_USD', 
    'total_production_tonnes_2020': 'total_production_t'
}

predictors = predictors.rename(columns=rename_dict)

In [117]:
# save
predictors.to_parquet(save_path_predictors_abs, index=False)

### Create relative dataset

In [118]:
##### Determine ISO3 of each pixel 

# Create geometry from pixel centroids
predictors['geometry'] = [Point(xy) for xy in zip(predictors['lon'], predictors['lat'])]

# Convert to GeoDataFrame (use same CRS as countries, which should be EPSG:4326)
countries = countries.to_crs("EPSG:4326")
gdf_predictors = gpd.GeoDataFrame(predictors, geometry='geometry', crs='EPSG:4326')

# Spatial join: each pixel gets the country it falls within
gdf_predictors = gpd.sjoin(gdf_predictors, countries[['geometry', 'GID_0']], how='left', predicate='within')

# Fill missing GID_0 with nearest country
missing_mask = gdf_predictors['GID_0'].isna()
if missing_mask.any():
    # Reproject to equal-area for distance calculations
    gdf_predictors_proj = gdf_predictors.to_crs("EPSG:6933") # convert to equal area for projection
    countries_proj = countries.to_crs("EPSG:6933")
    
    # Find nearest country for pixels with missing GID_0
    missing_pixels = gdf_predictors_proj[missing_mask].copy()
    nearest = gpd.sjoin_nearest(missing_pixels[['geometry']], 
                                 countries_proj[['geometry', 'GID_0']], 
                                 how='left')
    
    # Update GID_0 with nearest country
    gdf_predictors.loc[missing_mask, 'GID_0'] = nearest['GID_0'].values

In [119]:
##### Attatch country data 

# get correct ISO code 
gdf_predictors = gdf_predictors.merge(country_names[['ISO3', 'SHP_code']], left_on='GID_0', right_on='SHP_code', how='left')

# merge to country data
country_avg = country_avg.rename(columns=lambda c: c if c == 'ISO3' else f'country_{c}')

gdf_predictors = gdf_predictors.merge(country_avg, on='ISO3', how='left')


In [120]:
##### Do log transformations of all intensities and densities

log_columns = ['pop_density_people_per_km2', 'country_pop_density_people_per_km2', 'USD_production_per_HA', 'country_USD_production_per_HA',
       'tonnes_production_per_HA', 'country_tonnes_production_per_HA', 'buffalo_density_per_km2', 'country_buffalo_density_per_km2',
       'chicken_density_per_km2', 'country_chicken_density_per_km2', 'cattle_density_per_km2', 'country_cattle_density_per_km2',
       'goats_density_per_km2', 'country_goats_density_per_km2', 'pigs_density_per_km2', 'country_pigs_density_per_km2',
       'sheep_density_per_km2', 'country_sheep_density_per_km2', 'livestock_density_LU_per_km2', 'country_livestock_density_LU_per_km2',
       'country_labor_intensity_USD',
       'country_capital_intensity_USD',
       'country_labor_intensity_tonnes',
       'country_capital_intensity_tonnes']

for col in log_columns:
    gdf_predictors[f"log_{col}"] = np.log1p(gdf_predictors[col])
    gdf_predictors = gdf_predictors.drop(columns=[col])


In [121]:
##### Calculate relative variables 

# log variables 
log_columns = ['pop_density_people_per_km2', 'USD_production_per_HA',
       'tonnes_production_per_HA', 'buffalo_density_per_km2',
       'chicken_density_per_km2', 'cattle_density_per_km2',
       'goats_density_per_km2', 'pigs_density_per_km2',
       'sheep_density_per_km2', 'livestock_density_LU_per_km2']

for col in log_columns:
    gdf_predictors[f"rtv_log_{col}"] = gdf_predictors[f'log_{col}'] - gdf_predictors[f"log_country_{col}"]
    gdf_predictors = gdf_predictors.drop(columns=[f'log_{col}', f"log_country_{col}"])

# other variables 
non_log_var = ['SOC', 'pct_cropland_irrigated', 'female_share','average_travel_time_city',
       'average_travel_time_port', 'probability_economic_land_use_objective',
       'probability_survival_land_use_objective', 'growing_season_length_days',
       'child_dependency_ratio', 'GDP_pc', 'slope', 
       'cereals_share_production_USD', 'fibres_share_production_USD',
       'fruits_share_production_USD', 'oilcrops_share_production_USD',
       'pulses_share_production_USD', 'roots_tubers_share_production_USD',
       'rest_of_crops_share_production_USD',
       'sugar_crops_share_production_USD', 'vegetables_share_production_USD',
       'rubber_share_production_USD', 'ruminants_share_production_USD',
       'monogastrics_share_production_USD', 'poultry_share_production_USD',
       'pct_GDP_ag', 'share_large_field', 'share_medium_field', 'share_small_field',
       'share_vsmall_field', 'share_with_nightlights',
       'crop_intensity']

for col in non_log_var:
    denom = gdf_predictors[f"country_{col}"].replace(0, np.nan)
    gdf_predictors[f"rtv_{col}"] = (gdf_predictors[col] / denom).fillna(0)
gdf_predictors = gdf_predictors.drop(columns=[col for col in non_log_var] + [f"country_{col}" for col in non_log_var])

# drop unneeeded cols
col_to_drop_cap = ['lon', 'lat', 'geometry', 'index_right',
       'GID_0', 'SHP_code', 'country_other_share_production_USD',
       'country_share_vlarge_field']

gdf_predictors = gdf_predictors.drop(columns=col_to_drop_cap)

In [122]:
### Save
gdf_predictors.to_parquet(save_path_predictors_rtv, index=False)